# 2026-07-28 atlas + time since fire (CyVerse)

Reads **MESMA outputs** from:

`new_data/2026-07-28/mesma/<tile>/`

Example (what you already have):

`new_data/2026-07-28/mesma/11_S_KR_2026_7_28/11_S_KR_2026_7_28_mesma_..._fractions`

Upload these **four files** into the same VICE folder as this notebook (overwrite old copies):
- `cyverse_pipeline_2026_07_28.ipynb` (this notebook)
- `export_time_since_fire.py`
- `build_atlas_cyverse.py`
- `postfire_landscape_atlas1.html`

Do **not** upload the GB MESMA cubes — they already live in `new_data/2026-07-28/mesma/`.

Run cells **in order**. Set `TILE` in the next cell to pick one scene (`11_S_KR`) or leave `'all'`. After the atlas builds, open `atlas/index.html` — scenes are **clickable official ESA tile footprints** (same KML as the Copernicus matplotlib plot). Also upload the new `build_atlas_cyverse.py`.

In [69]:
from pathlib import Path

DATE = '2026-07-28'
# Which scene to analyse: 'all' or a folder / substring, e.g. '11_S_KR'
TILE = 'all'
cwd = Path.cwd().resolve()

def find_mesma_dir():
    checks = []
    for parent in [cwd, *cwd.parents]:
        checks.append(parent / 'new_data' / DATE / 'mesma')
        checks.append(parent / DATE / 'mesma')
        if parent.name == 'mesma' and parent.parent.name == DATE:
            checks.append(parent)
    checks.extend([
        Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data') / DATE / 'mesma',
        Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data') / DATE / 'mesma',
        Path('/home/jovyan/data-store/SCE-CU/sentinel2_data/new_data') / DATE / 'mesma',
        Path('/data-store/SCE-CU/sentinel2_data/new_data') / DATE / 'mesma',
    ])
    for p in checks:
        if p.exists():
            return p
    raise FileNotFoundError('Could not find new_data/2026-07-28/mesma')

MESMA = find_mesma_dir()
DATE_DIR = MESMA.parent
NEW = DATE_DIR.parent
TSF = NEW / 'Time_since_fire'

print('MESMA folder:', MESMA)
print()
tile_dirs = sorted(p for p in MESMA.iterdir() if p.is_dir())
print(f'Tile folders: {len(tile_dirs)}')
fracs = []
for tile in tile_dirs:
    cubes = [p for p in tile.iterdir() if p.is_file() and p.name.endswith('_fractions') and p.suffix == '']
    print(f'  {tile.name}')
    for c in cubes:
        print(f'      {c.name}  ({c.stat().st_size/1e9:.1f} GB)')
        fracs.append(c)
print(f'\nFractions files listed: {len(fracs)}')
by_tile = {}
for c in fracs:
    by_tile.setdefault(c.parent.name, c)
fracs = [by_tile[k] for k in sorted(by_tile)]
selected = fracs
print(f'Unique tiles to convert: {len(selected)}')
print('TILE setting:', TILE)
print('TSF folder:', TSF, 'exists' if TSF.exists() else 'MISSING')
print('Already built:')
atlas_dir = DATE_DIR / 'atlas'
for c in selected:
    done = (atlas_dir / c.parent.name / 'mesma_grid.js').exists()
    print(f'  {c.parent.name}  {"SKIP" if done else "BUILD"}')

MESMA folder: /home/jovyan/data-store/new_data/2026-07-28/mesma

Tile folders: 42
  10_S_GB_2026_7_28
      10_S_GB_2026_7_28_mesma_20260830T203227_fractions  (6.8 GB)
  11_S_KR_2026_7_28
      11_S_KR_2026_7_28_mesma_20260830T211316_fractions  (6.8 GB)
  11_S_KS_2026_7_28
      11_S_KS_2026_7_28_mesma_20260831T013621_fractions  (6.8 GB)
  11_S_KT_2026_7_28
      11_S_KT_2026_7_28_mesma_20260831T045048_fractions  (6.8 GB)
  11_S_KU_2026_7_28
      11_S_KU_2026_7_28_mesma_20260831T065633_fractions  (6.8 GB)
  11_S_KV_2026_7_28
      11_S_KV_2026_7_28_mesma_20260831T075029_fractions  (6.8 GB)
  11_S_LA_2026_7_28
      11_S_LA_2026_7_28_mesma_20260831T080145_fractions  (6.8 GB)
  11_S_LB_2026_7_28
      11_S_LB_2026_7_28_mesma_20260831T104710_fractions  (6.8 GB)
  11_S_LR_2026_7_28
      11_S_LR_2026_7_28_mesma_20260831T124321_fractions  (6.8 GB)
  11_S_LS_2026_7_28
      11_S_LS_2026_7_28_mesma_20260831T172729_fractions  (6.8 GB)
  11_S_LT_2026_7_28
      11_S_LT_2026_7_28_mesma_20260831

## 1. Export time-since-fire GeoTIFF

Writes MODIS days-since-burn through **2026-07-28** into `new_data/Time_since_fire/`.
Authenticate Earth Engine if prompted (project `timesincefire`).

In [70]:
from pathlib import Path
import shutil

atlas = DATE_DIR / 'atlas'
atlas.mkdir(parents=True, exist_ok=True)

search = [
    Path.cwd(),
    Path.cwd() / 'new_data' / 'Time_since_fire',
    TSF,
    Path('/home/jovyan/data-store/new_data/Time_since_fire'),
    Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire'),
    Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire'),
    Path('/home/jovyan/data-store'),
]

print('Searching for burn map files:')
for d in search:
    h = d / 'southern_california_burn_map.html'
    p = d / 'southern_california_burn_map.png'
    print(f'  {d}')
    print(f'    html={h.exists()}  png={p.exists()}')

png = next(
    (d / 'southern_california_burn_map.png' for d in search
     if (d / 'southern_california_burn_map.png').exists()),
    None,
)
html = None
if png is not None and png.with_suffix('.html').exists():
    html = png.with_suffix('.html')
if html is None:
    html = next(
        (d / 'southern_california_burn_map.html' for d in search
         if (d / 'southern_california_burn_map.html').exists()),
        None,
    )
if html is None:
    raise FileNotFoundError(
        'southern_california_burn_map.html not found under SCE-CU Time_since_fire or where you ran method2.py'
    )

shutil.copy2(html, atlas / 'southern_california_burn_map.html')
print(f'copied html  {html.stat().st_size/1e6:.2f} MB  from {html}')
if png is not None:
    shutil.copy2(png, atlas / 'southern_california_burn_map.png')
    print(f'copied png   {png.stat().st_size/1e6:.2f} MB  from {png}')
    print('Using method2 static PNG (fire colors should stay).')
else:
    print('HTML only (old Earth Engine tiles). If fire colors are missing, put southern_california_burn_map.png next to the HTML and re-run this cell.')
print('Then Cmd+Shift+R on the atlas tab.')

Searching for burn map files:
  /home/jovyan/data-store
    html=False  png=False
  /home/jovyan/data-store/new_data/Time_since_fire
    html=True  png=True
  /home/jovyan/data-store/new_data/Time_since_fire
    html=True  png=True
  /home/jovyan/data-store/new_data/Time_since_fire
    html=True  png=True
  /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire
    html=True  png=False
  /home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire
    html=False  png=False
  /home/jovyan/data-store
    html=False  png=False
copied html  0.11 MB  from /home/jovyan/data-store/new_data/Time_since_fire/southern_california_burn_map.html
copied png   0.08 MB  from /home/jovyan/data-store/new_data/Time_since_fire/southern_california_burn_map.png
Using method2 static PNG (fire colors should stay).
Then Cmd+Shift+R on the atlas tab.


## 1b. Clickable tile map (no MESMA rebuild)

Rewrites `atlas/index.html` so each Sentinel-2 scene is a **clickable official ESA footprint** (same `S2A_OPER_GIP_TILPAR` KML as `download_from_copernicus_final.ipynb`). The red box is that notebook’s AOI:

`(-120.708008, 32.53)` to `(-113.975687, 38.395343)`

Upload the **new** `build_atlas_cyverse.py` next to this notebook. Run inventory (cell 1), then this cell. Skip the build cell if tiles are already built.

In [ ]:
from pathlib import Path
import importlib.util

atlas = DATE_DIR / 'atlas'
if not atlas.exists():
    raise FileNotFoundError('No atlas folder yet — run the build cell first')

build_py = Path.cwd() / 'build_atlas_cyverse.py'
if not build_py.exists():
    build_py = Path('/home/jovyan/data-store/build_atlas_cyverse.py')
if not build_py.exists():
    raise FileNotFoundError('Upload the new build_atlas_cyverse.py next to this notebook')

spec = importlib.util.spec_from_file_location('bac', build_py)
bac = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bac)

folders = [
    p.name for p in sorted(atlas.iterdir())
    if p.is_dir() and p.name != 'test' and (p / 'mesma_grid.js').exists()
]
bac.write_atlas_index(atlas, folders, mesma_dir=MESMA)
print('Open atlas/index.html (or the Open cell). Hard-refresh Cmd+Shift+R.')


## 2. Build the atlas

Uses `TILE` from cell 1 (`'all'` or e.g. `'11_S_KR'`). Writes small files to:

`new_data/2026-07-28/atlas/`

- `index.html` — pick a scene
- `<tile>/` — HTML + JS + PNG for that image

Download those small files — not the GB cubes. In the atlas UI, use **Scene to analyse** to switch tiles.

In [72]:
from IPython import get_ipython

html = Path.cwd() / 'postfire_landscape_atlas1.html'
if not html.exists():
    html = Path('/home/jovyan/data-store/postfire_landscape_atlas1.html')
to_build = []
for frac in selected:
    out = DATE_DIR / 'atlas' / frac.parent.name
    if (out / 'mesma_grid.js').exists():
        print('skip', frac.parent.name)
        continue
    to_build.append(frac)
print(f'Building {len(to_build)} remaining scene(s) of {len(selected)} unique tiles')

for frac in to_build:
    out = DATE_DIR / 'atlas' / frac.parent.name
    cmd = (
        f'build_atlas_cyverse.py --fractions {frac} --tsf-dir {TSF} '
        f'--html {html} --out {out}'
    )
    print('\n====', frac.parent.name, '====')
    get_ipython().run_line_magic('run', cmd)

# refresh HTML + scene picker in every built folder
html_src = html if html.exists() else None
atlas = DATE_DIR / 'atlas'
if html_src and atlas.exists():
    text = html_src.read_text()
    for d in sorted(p for p in atlas.iterdir() if p.is_dir()):
        if (d / 'mesma_grid.js').exists():
            (d / 'postfire_landscape_atlas1.html').write_text(text)
    print('Copied latest HTML into', sum(1 for p in atlas.iterdir() if p.is_dir()), 'tile folders')

print('\nAtlas outputs:')
if atlas.exists():
    print('tiles:', len([p for p in atlas.iterdir() if p.is_dir()]))
    print('size MB:', round(sum(p.stat().st_size for p in atlas.rglob('*') if p.is_file())/1e6, 1))
else:
    print('No atlas folder yet')

skip 10_S_GB_2026_7_28
skip 11_S_KR_2026_7_28
skip 11_S_KS_2026_7_28
skip 11_S_KT_2026_7_28
skip 11_S_KU_2026_7_28
skip 11_S_KV_2026_7_28
skip 11_S_LA_2026_7_28
skip 11_S_LB_2026_7_28
skip 11_S_LR_2026_7_28
skip 11_S_LS_2026_7_28
skip 11_S_LT_2026_7_28
skip 11_S_LU_2026_7_28
skip 11_S_LV_2026_7_28
skip 11_S_MA_2026_7_28
skip 11_S_MB_2026_7_28
skip 11_S_MR_2026_7_28
skip 11_S_MS_2026_7_28
skip 11_S_MT_2026_7_28
skip 11_S_MU_2026_7_28
skip 11_S_MV_2026_7_28
skip 11_S_NA_2026_7_28
skip 11_S_NB_2026_7_28
skip 11_S_NR_2026_7_28
skip 11_S_NS_2026_7_28
skip 11_S_NT_2026_7_28
skip 11_S_NU_2026_7_28
skip 11_S_NV_2026_7_28
skip 11_S_PA_2026_7_28
skip 11_S_PB_2026_7_28
skip 11_S_PR_2026_7_28
skip 11_S_PS_2026_7_28
skip 11_S_PT_2026_7_28
skip 11_S_PV_2026_7_28
skip 11_S_QA_2026_7_28
skip 11_S_QB_2026_7_28
skip 11_S_QR_2026_7_28
skip 11_S_QS_2026_7_28
skip 11_S_QT_2026_7_28
skip 11_S_QU_2026_7_28
skip 11_S_QV_2026_7_28
skip test
Building 1 remaining scene(s) of 42 unique tiles

==== 11_S_PU_2026_7_

SystemExit: No modeled MESMA pixels found (shade band is empty).

Copied latest HTML into 43 tile folders

Atlas outputs:
tiles: 43
size MB: 309.8


## Open the HTML on CyVerse (do not download the GB cubes)

The `atlas/` folder is only ~20 MB. Jupyter's built-in HTML preview will look empty — it blocks `mesma_grid.js`.

Run the next cell for clickable links, **or** in the file browser: open a tile folder → right-click `postfire_landscape_atlas1.html` → **Open in New Browser Tab**.

In [73]:
from pathlib import Path

needles = [
    Path.cwd() / 'postfire_landscape_atlas1.html',
    Path('/home/jovyan/data-store/postfire_landscape_atlas1.html'),
    Path('/home/jovyan/postfire_landscape_atlas1.html'),
]
src = next((p for p in needles if p.exists() and 'zoomFitBtn' in p.read_text()), None)
print('source:', src)
if src is None:
    raise SystemExit('New HTML not found. Upload postfire_landscape_atlas1.html into the same folder as the notebook.')

atlas = Path('/home/jovyan/data-store/new_data/2026-07-28/atlas')
text = src.read_text()
for d in sorted(atlas.iterdir()):
    dest = d / 'postfire_landscape_atlas1.html'
    if d.is_dir() and dest.exists():
        dest.write_text(text)
        print(d.name, 'now has Fit:', 'zoomFitBtn' in dest.read_text())

source: /home/jovyan/data-store/postfire_landscape_atlas1.html
10_S_GB_2026_7_28 now has Fit: True
11_S_KR_2026_7_28 now has Fit: True
11_S_KS_2026_7_28 now has Fit: True
11_S_KT_2026_7_28 now has Fit: True
11_S_KU_2026_7_28 now has Fit: True
11_S_KV_2026_7_28 now has Fit: True
11_S_LA_2026_7_28 now has Fit: True
11_S_LB_2026_7_28 now has Fit: True
11_S_LR_2026_7_28 now has Fit: True
11_S_LS_2026_7_28 now has Fit: True
11_S_LT_2026_7_28 now has Fit: True
11_S_LU_2026_7_28 now has Fit: True
11_S_LV_2026_7_28 now has Fit: True
11_S_MA_2026_7_28 now has Fit: True
11_S_MB_2026_7_28 now has Fit: True
11_S_MR_2026_7_28 now has Fit: True
11_S_MS_2026_7_28 now has Fit: True
11_S_MT_2026_7_28 now has Fit: True
11_S_MU_2026_7_28 now has Fit: True
11_S_MV_2026_7_28 now has Fit: True
11_S_NA_2026_7_28 now has Fit: True
11_S_NB_2026_7_28 now has Fit: True
11_S_NR_2026_7_28 now has Fit: True
11_S_NS_2026_7_28 now has Fit: True
11_S_NT_2026_7_28 now has Fit: True
11_S_NU_2026_7_28 now has Fit: True
1

In [74]:
import os
from IPython.display import display, HTML

atlas = DATE_DIR / 'atlas'
size_mb = sum(p.stat().st_size for p in atlas.rglob('*') if p.is_file()) / 1e6
print(f'atlas/ size: {size_mb:.1f} MB  (this is what the HTML needs — not the GB MESMA cubes)')

prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
if not prefix.endswith('/'):
    prefix += '/'

roots = []
for r in (Path.cwd(), Path('/home/jovyan/data-store'), Path('/home/jovyan')):
    try:
        roots.append((r, atlas.resolve().relative_to(r.resolve())))
    except ValueError:
        pass
rel = roots[0][1] if roots else Path(atlas.name)
base = f'{prefix}files/{rel.as_posix()}'

rows = [f'<p><b>Open in a new browser tab</b> (not Jupyter preview):</p>']
rows.append(f'<p><a href="{base}/index.html" target="_blank">Scene picker (index.html)</a></p><ul>')
for folder in sorted(p for p in atlas.iterdir() if p.is_dir()):
    html_name = folder / 'postfire_landscape_atlas1.html'
    if html_name.exists():
        rows.append(f'<li><a href="{base}/{folder.name}/postfire_landscape_atlas1.html" target="_blank">{folder.name}</a></li>')
rows.append('</ul>')
rows.append('<p>If a link 404s, in the file browser go into the tile folder, right-click the HTML → Open in New Browser Tab.</p>')
display(HTML('\n'.join(rows)))

atlas/ size: 309.8 MB  (this is what the HTML needs — not the GB MESMA cubes)


In [75]:
import os, socket, subprocess, time
from IPython.display import display, HTML

atlas = DATE_DIR / 'atlas'
PORT = 8765
size_mb = sum(p.stat().st_size for p in atlas.rglob('*') if p.is_file()) / 1e6
print(f'atlas/ is {size_mb:.1f} MB (safe to serve here; do not download the GB cubes)')

def port_open(port):
    with socket.socket() as s:
        s.settimeout(0.3)
        return s.connect_ex(('127.0.0.1', port)) == 0

if not atlas.exists():
    raise FileNotFoundError('No atlas folder yet — run the build cell first')

if not port_open(PORT):
    subprocess.Popen(
        ['python3', '-m', 'http.server', str(PORT), '--bind', '127.0.0.1'],
        cwd=str(atlas),
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(0.8)

prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
if not prefix.endswith('/'):
    prefix += '/'
url = f'{prefix}proxy/{PORT}/index.html'
print('Click this link (new browser tab):')
print(url)
display(HTML(
    f'<p style="font-size:18px"><a href="{url}" target="_blank" rel="noopener">'
    f'<b>Open postfire atlas</b></a></p>'
    f'<p>You should see 5 scene cards. If this 404s, Jupyter proxy is off; '
    f'try the printed URL or ask for jupyter-server-proxy.</p>'
))

atlas/ is 309.8 MB (safe to serve here; do not download the GB cubes)
Click this link (new browser tab):
/proxy/8765/index.html


In [76]:
from pathlib import Path

needles = [
    Path.cwd() / 'postfire_landscape_atlas1.html',
    Path('/home/jovyan/data-store/postfire_landscape_atlas1.html'),
    Path('/home/jovyan/postfire_landscape_atlas1.html'),
]
src = next((p for p in needles if p.exists() and 'zoomFitBtn' in p.read_text()), None)
print('source:', src)
if src is None:
    raise SystemExit('New HTML not found. Upload postfire_landscape_atlas1.html into the same folder as the notebook.')

atlas = Path('/home/jovyan/data-store/new_data/2026-07-28/atlas')
text = src.read_text()
for d in sorted(atlas.iterdir()):
    dest = d / 'postfire_landscape_atlas1.html'
    if d.is_dir() and dest.exists():
        dest.write_text(text)
        print(d.name, 'now has Fit:', 'zoomFitBtn' in dest.read_text())

source: /home/jovyan/data-store/postfire_landscape_atlas1.html
10_S_GB_2026_7_28 now has Fit: True
11_S_KR_2026_7_28 now has Fit: True
11_S_KS_2026_7_28 now has Fit: True
11_S_KT_2026_7_28 now has Fit: True
11_S_KU_2026_7_28 now has Fit: True
11_S_KV_2026_7_28 now has Fit: True
11_S_LA_2026_7_28 now has Fit: True
11_S_LB_2026_7_28 now has Fit: True
11_S_LR_2026_7_28 now has Fit: True
11_S_LS_2026_7_28 now has Fit: True
11_S_LT_2026_7_28 now has Fit: True
11_S_LU_2026_7_28 now has Fit: True
11_S_LV_2026_7_28 now has Fit: True
11_S_MA_2026_7_28 now has Fit: True
11_S_MB_2026_7_28 now has Fit: True
11_S_MR_2026_7_28 now has Fit: True
11_S_MS_2026_7_28 now has Fit: True
11_S_MT_2026_7_28 now has Fit: True
11_S_MU_2026_7_28 now has Fit: True
11_S_MV_2026_7_28 now has Fit: True
11_S_NA_2026_7_28 now has Fit: True
11_S_NB_2026_7_28 now has Fit: True
11_S_NR_2026_7_28 now has Fit: True
11_S_NS_2026_7_28 now has Fit: True
11_S_NT_2026_7_28 now has Fit: True
11_S_NU_2026_7_28 now has Fit: True
1